# Predicting buying pressure on a security, and testing it for alpha

A **security-level** study, separate from the fund-level replication.

| | |
|---|---|
| unit of observation | (security, quarter) |
| **target** | `buy_frac(s, q+1)` = # funds buying s / # funds owning s, next quarter |
| **features** | latest Barra `GEMLT_*` exposures at quarter end, past security returns, lagged buying pressure and ownership breadth |
| **alpha test** | rank securities by *predicted* buying pressure, then look at forward returns |

## The timing trap, stated once

`buy_frac` over the window q → q+1 compares holdings at q against holdings at q+1, so it is
**not observable at q**. It is known once the q+1 holdings exist, and public only after the
~45–60 day filing delay.

That has two consequences the code enforces:

- **only LAGGED buy_frac may be a feature** — `buy_frac_lag1` refers to a window that closed
  before the target window opened. The contemporaneous `buy_frac` is *not* in the feature set.
- **returns must start after the target window**, otherwise the sort variable and the return
  share the same quarter:

| | return window | status |
|---|---|---|
| `contemporaneous` | overlaps the target window | **biased**, benchmark only |
| `predictive` | starts when the target window opens | no overlap, ignores the filing lag ← default |
| `tradeable` | one further quarter out | also clears the filing lag |

All three are always reported.

## Joining Barra to the holdings

Barra is keyed by **`sedol`**, the holdings panel by its own **`security`** id. They only
join through a mapping:

- if the holdings file already carries a sedol → set `holdings_sedol_col`
- otherwise → point `sedol_map_path` at a table with both ids

The loader prints the match rate and **refuses to run** below `min_match_rate`, so a broken
mapping fails loudly instead of producing an empty panel.

## Models

`gbm` (history via explicit lags), `lstm` (one sample = a security's last 8 quarters `[T, F]`),
`ridge` (linear benchmark). Same rolling split for all three.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from dataclasses import replace
%load_ext autoreload
%autoreload 2
import buy_pressure as B
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
B.check_version()

## 0. Inspect the Barra pickle first

Before anything else, look at the actual structure — column names, how the `GEMLT_` factors
are laid out, and whether `day` / `sedol` are columns or part of the index.

In [ ]:
raw = B.inspect_barra(B.Config())

## 1. Configuration

Set `holdings_sedol_col` **or** `sedol_map_path` — without one of them Barra cannot be
joined and the loader will say so.

In [ ]:
CFG_KW = dict(
    holdings_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    barra_path    = "manager_holdings/barra_GEMLTL_R3000_Prod_new_MSCI_wkly.pickle",

    # --- how the two datasets join (set ONE of these) ---
    holdings_sedol_col = None,      # e.g. "sedol" if the holdings file already has it
    sedol_map_path     = None,      # else a parquet/csv holding both ids
    map_security_col   = "security",
    map_sedol_col      = "sedol",

    inv_type_codes = (401,),
    max_rank    = None,             # None = whole book; buying pressure is a breadth
                                    # measure, so truncating the book distorts it
    min_owners  = 5,                # a buy_frac from 2 funds is noise
    min_quarters = 12,
    eval_timing = "predictive",
    window_q = 28, test_q = 8, step = 8,
    model = "gbm",
    seq_len = 8, hidden = 64, max_epochs = 40, batch = 4096, device = "auto",
)
known = set(B.Config.__dataclass_fields__)
dropped = {k: v for k, v in CFG_KW.items() if k not in known}
BASE = B.Config(**{k: v for k, v in CFG_KW.items() if k in known})
print("!! unsupported by this copy:", list(dropped)) if dropped else print("all keys accepted")
BASE

## 2. Build the panel

Watch three printed diagnostics:

- **`[hold] buy_frac mean / sd`** — if sd is tiny there is no cross-sectional variation to predict
- **`[hold] median owners`** — with few owners per security the target is mostly noise; raise `min_owners`
- **`[join] Barra matched on X%`** — the id mapping. Anything near 0 means the join is wrong

In [ ]:
panel = B.build_panel(BASE)
feats = B.feature_list(panel, BASE)
print(f"\n{len(feats)} features")
print("  Barra exposures :", [f for f in feats if f.startswith(BASE.barra_prefix)][:12])
print("  buying history  :", [f for f in feats if "buy_frac" in f or "sell_frac" in f or "owning" in f])
print("  returns / other :", [f for f in feats if f.startswith("ret_") or f.startswith("log_") or f == "w_mean"])

In [ ]:
display(panel[["buy_frac", "sell_frac", "n_owning", "target_buy_frac"]]
        .describe().loc[["count","mean","std","min","25%","50%","75%","max"]].round(4))
print("target availability by quarter (tail):")
display(panel.groupby("yq")["target_buy_frac"].agg(["size", "mean"]).tail(8).round(4))

---
## 3. GBM

In [ ]:
RESULTS = {}
RESULTS["gbm"] = B.run_one(panel, replace(BASE, model="gbm"), "gbm")
B.free(RESULTS)

In [ ]:
r = RESULTS["gbm"]
print("### prediction quality (is next-quarter buying predictable at all?)")
display(r["quality"].round(4))
print("### alpha: securities ranked by PREDICTED buying pressure")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"-- {tm} --"); display(r["alpha_pred"][tm].round(3))
print("### reference: ranked by the CURRENT window's actual buying (not tradeable)")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"-- {tm} --"); display(r["alpha_actual"][tm].round(3))

---
## 4. LSTM

One sample is a security's last `seq_len` quarters of features. Sequences are assembled from
indices rather than materialised, so memory stays near the size of the feature matrix. Set
`lstm_max_train` if a CPU-only run drags.

In [ ]:
RESULTS["lstm"] = B.run_one(panel, replace(BASE, model="lstm"), "lstm")
B.free(RESULTS)

In [ ]:
r = RESULTS["lstm"]
display(r["quality"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"-- alpha, predicted buying / {tm} --"); display(r["alpha_pred"][tm].round(3))

---
## 5. Ridge — linear benchmark

If ridge and gbm land in the same place, the signal is broadly linear and the tree is not
inventing structure.

In [ ]:
RESULTS["ridge"] = B.run_one(panel, replace(BASE, model="ridge"), "ridge")
B.free(RESULTS)

In [ ]:
display(RESULTS["ridge"]["quality"].round(4))
display(RESULTS["ridge"]["alpha_pred"][BASE.eval_timing].round(3))

---
## 6. Comparison

Two separate questions, and they can have different answers:

1. **Is next-quarter buying predictable?** → `rank_IC` versus `naive_IC` (just persisting
   this window's buying pressure). If the model does not beat persistence, the features add nothing.
2. **Does predicted buying carry alpha?** → the `alpha_*` columns. A spread that exists only
   under `contemporaneous` is same-quarter co-movement, not a forecast.

In [ ]:
cmp = B.compare(RESULTS)
cmp

## Save

In [ ]:
import os
os.makedirs("outputs_buy_pressure", exist_ok=True)
for tag, r in RESULTS.items():
    if not isinstance(r, dict) or "quality" not in r: continue
    r["quality"].to_csv(f"outputs_buy_pressure/quality_{tag}.csv", index=False)
    for tm in ("predictive", "tradeable", "contemporaneous"):
        r["alpha_pred"][tm].to_csv(f"outputs_buy_pressure/alpha_pred_{tag}_{tm}.csv", index=False)
        r["alpha_actual"][tm].to_csv(f"outputs_buy_pressure/alpha_actual_{tag}_{tm}.csv", index=False)
cmp.to_csv("outputs_buy_pressure/compare_models.csv", index=False)
print("saved to outputs_buy_pressure/")

## Reading the result honestly

- **`rank_IC` near zero** → next-quarter buying pressure is not predictable from these
  features. That is a finding, not a failure; report it rather than tuning until something appears.
- **IC positive but no alpha** → buying is forecastable yet carries no return information.
- **alpha only under `contemporaneous`** → the spread lives in the same quarter as the
  target window; it is co-movement, not a forecast.
- **alpha under `tradeable`** → the only version that survives the filing delay.

Returns here are gross — no trading costs, and a buying-pressure sort implies turnover, so
any spread should be discounted before it is called alpha.